# LakeSoul 视频数据处理演示

### 1. 读取视频文件, 使用 daft 进行信息抽取
### 2. 写入 LakeSoul 多模态湖仓
### 3. 从湖仓中读取写入数据并展示

演示 python 环境，请选择 conda 中的 video  
请保证 LakeSoul 元数据库里，没有相同的表

In [1]:
import logging
logging.disable(logging.CRITICAL)

In [13]:
from IPython.display import Video
video_path = "/home/maji/Projects/multi/video/top.mp4"
# Video(video_path, width=800)
Video(video_path)

 使用 `daft.read_video_frames` 抽取视频中的关键帧

In [2]:
import daft
from daft import col, DataType
from daft.functions import encode_image

video_path = "/home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi"
# 设置 Daft 执行器
daft.set_runner_ray(noop_if_initialized=True)
df = daft.read_video_frames(
    path=video_path,
    image_height=480,
    image_width=640,
    is_key_frame=True,
    sample_interval_seconds=1.0,
)
df = df.with_column( "video_path", daft.lit(video_path)).with_column("data", encode_image(col("data"), "JPEG"))

df.show()



pathString,frame_indexInt64,frame_timeFloat64,frame_time_baseString,frame_ptsInt64,frame_dtsInt64,frame_durationInt64,is_key_frameBool,dataBinary,video_pathString
file:///home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi,0,0,1001/30000,0,0,1,true,"b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01""...",/home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi
file:///home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi,3,1.2012,1001/30000,36,36,1,true,"b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01""...",/home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi
file:///home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi,5,2.002,1001/30000,60,60,1,true,"b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01""...",/home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi
file:///home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi,8,3.2032,1001/30000,96,96,1,true,"b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01""...",/home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi
file:///home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi,10,4.004,1001/30000,120,120,1,true,"b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01""...",/home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi


使用 `create_table` 创建 LakeSoul 表，并写入

In [7]:
from lakesoul.metadata import create_table
from lakesoul.ray import LakeSoulDatasink
schema = df.schema().to_pyarrow_schema()
create_table(
    "video_frames_table",
    table_schema=schema,
    table_path="/home/maji/Projects/multi/video/video_frames_table",
)
ds = df.to_ray_dataset()
sink = LakeSoulDatasink("video_frames_table")
ds.write_datasink(sink)

使用 `ray.data.read_lakesoul()` 读取 LakeSoul 表  
之后将 `ray.data.Dataset` 转换成  `daft.dataframe`  
将关键帧 data 解码为 image 

In [8]:
import ray
import lakesoul.ray
from daft import col
from daft.functions import decode_image

df = ray.data.read_lakesoul("video_frames_table").to_daft()

df = df.with_column(
    "data",
    decode_image(col("data"))
)

# data 放到第一列
cols = df.column_names

df = df.select(
    "data",
    *[c for c in cols if c != "data"]
)

df.show()

(pid=490988) InMemoryScan->DistributedLimit->Project:   0%|          | 0.00/1.00 [00:00<?, ?it/s]

dataImage[RGB],pathString,frame_indexInt64,frame_timeFloat64,frame_time_baseString,frame_ptsInt64,frame_dtsInt64,frame_durationInt64,is_key_frameBool,video_pathString
""" />",file:///home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi,10,4.004,1001/30000,120,120,1,true,/home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi
""" />",file:///home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi,0,0,1001/30000,0,0,1,true,/home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi
""" />",file:///home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi,3,1.2012,1001/30000,36,36,1,true,/home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi
""" />",file:///home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi,5,2.002,1001/30000,60,60,1,true,/home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi
""" />",file:///home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi,8,3.2032,1001/30000,96,96,1,true,/home/maji/Projects/multi/video/data/UCF101_subset/train/Basketball/v_Basketball_g01_c01.avi
